In [3]:
import pandas as pd
from sqlalchemy import create_engine, text
import os

# ====================== CONNECTION ======================
# Apna connection string yahan daal do
DB_USER = "postgres"
DB_PASSWORD = "postgres"        # ← Change this
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "warehouse"                # ← Change if different

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# Test connection
try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("✅ Database Connection Successful!\n")
except Exception as e:
    print("❌ Connection Failed:", e)
    raise

# ====================== HELPER FUNCTION ======================
def load_csv_to_bronze(source_folder, csv_file, table_name):
    """Load CSV into bronze schema"""
    file_path = os.path.join("datasets", source_folder, csv_file)
    
    print(f"Loading: {csv_file} ...", end=" ")
    
    df = pd.read_csv(file_path, encoding_errors='ignore')
    
    # Load into bronze schema
    df.to_sql(
        name=table_name,
        con=engine,
        schema="bronze",
        if_exists="replace",      # ya "append" if you want
        index=False
    )
    
    print(f"✅ Done | Rows: {len(df)}")

# ====================== CREATE BRONZE SCHEMA ======================
with engine.connect() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS bronze;"))
    conn.commit()
print("✅ Bronze schema ready\n")

# ====================== LOAD ALL FILES ======================

# --- CRM Source ---
load_csv_to_bronze("source_crm", "cust_info.csv",      "crm_cust_info")
load_csv_to_bronze("source_crm", "prd_info.csv",      "crm_prd_info")
load_csv_to_bronze("source_crm", "sales_details.csv", "crm_sales_details")

# --- ERP Source ---
load_csv_to_bronze("source_erp", "CUST_AZ12.csv",     "erp_cust_az12")
load_csv_to_bronze("source_erp", "loc_a101.csv",      "erp_loc_a101")
load_csv_to_bronze("source_erp", "PX_CAT_G1V2.csv",   "erp_px_cat_g1v2")

print("\n🎉 All 6 CSV files loaded successfully into bronze schema!")

✅ Database Connection Successful!

✅ Bronze schema ready

Loading: cust_info.csv ... ✅ Done | Rows: 18494
Loading: prd_info.csv ... ✅ Done | Rows: 397
Loading: sales_details.csv ... ✅ Done | Rows: 60398
Loading: CUST_AZ12.csv ... ✅ Done | Rows: 18484
Loading: loc_a101.csv ... ✅ Done | Rows: 18484
Loading: PX_CAT_G1V2.csv ... ✅ Done | Rows: 37

🎉 All 6 CSV files loaded successfully into bronze schema!
